# data cleaning

In [50]:
%pip install pandas numpy matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.preprocessing import StandardScaler

In [46]:
train = pd.read_csv('train_(2)_(1)_(1)_(1).csv')
test = pd.read_csv('test_(2)_(1)_(1)_(1).csv')
avgRent = pd.read_csv('avg_rent_(1)_(1)_(1)_(1).csv')
disCityCenter = pd.read_csv('dist_from_city_centre_(1)_(1)_(1)_(1).csv')
sampleSubmission = pd.read_csv('sample_submission_(3)_(1)_(1)_(1).csv')

In [47]:
# print(train.info())
train.dropna(subset=['size', 'location', 'bath', 'balcony'], inplace=True)
# print(train.info())


### cleaning total_sqft and making it numeric 
as it is expected to be in sq feet

then scaling it

In [48]:
def is_number(x):
    try:
        float(x)
        return True
    except ValueError:
        return False

def return_sqft(x):
    if is_number(x):
        return float(x)
    
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    if x.endswith('Sq. Meter') or x.endswith('Sq. Meter'):
        x = float(x.split('Sq')[0])
        return x*10.7639
    if x.endswith('Acres'):
        x = float(x.split('Acr')[0])
        return x*43560
    if x.endswith('Sq. Yards'):
        x = float(x.split('Sq')[0])
        return x*10.7639
    if x.endswith('Grounds'):
        x = float(x.split('Gro')[0])
        return x*2400
    if x.endswith('Cents'):
        x = float(x.split('Cen')[0])
        return x*435.6
    if x.endswith('Perch'):
        x = float(x.split('Per')[0])
        return x*272.25
    if x.endswith('Guntha'):
        x = float(x.split('Gun')[0])
        return x*1089
    
    print("Couldn't parse the value: ", x)
    return x



train['total_sqft'] = train.total_sqft.apply(lambda x: return_sqft(x))
train[train.total_sqft.apply(lambda x: is_number(x) == False)]


train['price_per_sqft'] = train.price / train.total_sqft
# train.total_sqft.describe()

outlier treatment for total_sqft based on area_type


OUTLIERS HAVE NOT YET BEEN TREATED AND MUST BE TREATED FOR ALL CATEGORIES

In [49]:
# # FOR SOME REASON THE R^2 IS REDUCING
# # ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ HENCE THE CODE IS COMMENTED OUT ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

# # TODO: outlier analysis to be done wrt area type 

# # plt.subplots = plt.subplots(figsize=(10,5))


# print(train.area_type.value_counts())
# plt.figure(figsize=(20,10))


# plt.subplot(2,4,1)
# plt.title('Price per sqft distribution for Super built-up  Area')
# sns.boxplot(train[train.area_type == 'Super built-up  Area'].price_per_sqft)
# plt.subplot(2,4,5)
# plt.title('Price per sqft distribution for Super built-up  Area')
# sns.histplot(train[train.area_type == 'Super built-up  Area'].price_per_sqft)



# plt.subplot(2,4,2)
# plt.title('Price per sqft distribution for Built-up  Area')
# sns.boxplot(train[train.area_type == 'Built-up  Area'].price_per_sqft)
# plt.subplot(2,4,6)
# plt.title('Price per sqft distribution for Built-up  Area')
# sns.histplot(train[train.area_type == 'Built-up  Area'].price_per_sqft)



# plt.subplot(2,4,3)
# plt.title('Price per sqft distribution for Plot  Area')
# sns.boxplot(train[train.area_type == 'Plot  Area'].price_per_sqft)
# plt.subplot(2,4,7)
# plt.title('Price per sqft distribution for Plot  Area')
# sns.histplot(train[train.area_type == 'Plot  Area'].price_per_sqft)



# plt.subplot(2,4,4)
# plt.title('Price per sqft distribution for Carpet  Area')
# sns.boxplot(train[train.area_type == 'Carpet  Area'].price_per_sqft)
# plt.subplot(2,4,8)
# plt.title('Price per sqft distribution for Carpet  Area')
# sns.histplot(train[train.area_type == 'Carpet  Area'].price_per_sqft)



# plt.tight_layout()
# plt.show()


for i in train.area_type.unique():
    col = train[train.area_type == i].price_per_sqft
    iqr = (col.quantile(0.75) - col.quantile(0.25)) * (3/2)
    train.drop(train[(train.area_type == i) & (train.price_per_sqft > iqr)].index, inplace=True)


In [50]:
# print(train.price_per_sqft.describe())
# print(train[train.price_per_sqft > .6])
# train.drop(train[train.price_per_sqft > 0.6].index, inplace=True, axis=0)


# scaling data => total sq feet


In [51]:
# note that we must do outlier treatment before scaling
# min max scaling for total_sqft

# train.total_sqft = train.total_sqft.apply(lambda x: (x - train.total_sqft.min())/(train.total_sqft.max() - train.total_sqft.min()))

successfully cleaned all non numeric data from total_sqft col

## handling size
- currently 1rk = 0
- 1bhk/1bedroom = 1
- 2bhk/2bedroom = 2 
and soo on

In [52]:
# print(train['size'].value_counts(dropna=False))
# train[train['size'].apply(lambda x: is_number(x) == False)]

def returnSize(x):
    try:
        if x == '1 RK':
            return 0
        else:
            return int(x.split(' ')[0])
    except:
        print("Couldn't parse size: ", x)


train['size'] = train['size'].apply(lambda x: returnSize(x))

# train.info()



## avilibility


In [53]:
from datetime import datetime

# train.availability.value_counts()

def noOfDays(x):
    if x == 'Ready To Move':
        return 0
    tokens = x.split('-')
    if len(tokens) == 2:
        now = datetime.now()

        date = datetime.strptime(f'{x}-{now.year}', '%d-%b-%Y')

        if (date - now).days < 0:
            date = datetime.strptime(f'{x}-{now.year + 1}', '%d-%b-%Y')
            
        return (date - now).days
    else:
        print("Couldn't parse availability: ", x)
        return -1


train.availability = train.availability.apply(lambda x: noOfDays(x))
# train.info()


# standard scaling

In [54]:
scaler = StandardScaler()

numeric_cols = train.select_dtypes(include=[np.number]).columns
train[numeric_cols] = scaler.fit_transform(train[numeric_cols])

## encoding area type

n-1 dummy encoding performed 

with value counts

- area_type
- Super built-up  Area    6745
- Built-up  Area          1845
- Plot  Area              1496
- Carpet  Area              65

model performance to be tested including only Super built-up  Area 

In [55]:
temp = True
for i in train.area_type.unique():
    if temp:
        temp = False
        continue
    train["area_type_" + i.split("  ")[0].replace(" ", "_").replace("-", "_")] = train.area_type.apply(lambda x: 1 if x==i else 0)

train.drop('area_type', axis=1, inplace=True)

In [56]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2122 entries, 1 to 10651
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        2122 non-null   float64
 1   availability              2122 non-null   float64
 2   location                  2122 non-null   object 
 3   size                      2122 non-null   float64
 4   society                   831 non-null    object 
 5   total_sqft                2122 non-null   float64
 6   bath                      2122 non-null   float64
 7   balcony                   2122 non-null   float64
 8   price                     2122 non-null   float64
 9   price_per_sqft            2122 non-null   float64
 10  area_type_Super_built_up  2122 non-null   int64  
 11  area_type_Built_up        2122 non-null   int64  
 12  area_type_Carpet          2122 non-null   int64  
dtypes: float64(8), int64(3), object(2)
memory usage: 232.1+ KB


In [57]:
# import 'Pandas' 
import pandas as pd 

# import 'Numpy' 
import numpy as np

# import subpackage of Matplotlib
import matplotlib.pyplot as plt

# import 'Seaborn' 
import seaborn as sns

# to suppress warnings 
from warnings import filterwarnings
filterwarnings('ignore')

# display all columns of the dataframe
pd.options.display.max_columns = None

# display all rows of the dataframe
pd.options.display.max_rows = None
 
# to display the float values upto 6 decimal places     
pd.options.display.float_format = '{:.6f}'.format

# import train-test split 
from sklearn.model_selection import train_test_split

# # import various functions from statsmodels
# import statsmodels
import statsmodels.api as sm
# import statsmodels.stats.api as sms
# from statsmodels.graphics.gofplots import qqplot

# # import 'stats'
# from scipy import stats

# # 'metrics' from sklearn is used for evaluating the model performance
# from sklearn.metrics import mean_squared_error

# # import functions to perform feature selection
# from mlxtend.feature_selection import SequentialFeatureSelector as sfs
# #from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import RFE

# import function to perform linear regression
from sklearn.linear_model import LinearRegression

# import functions to perform cross validation
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [58]:
train.info()


<class 'pandas.core.frame.DataFrame'>
Index: 2122 entries, 1 to 10651
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        2122 non-null   float64
 1   availability              2122 non-null   float64
 2   location                  2122 non-null   object 
 3   size                      2122 non-null   float64
 4   society                   831 non-null    object 
 5   total_sqft                2122 non-null   float64
 6   bath                      2122 non-null   float64
 7   balcony                   2122 non-null   float64
 8   price                     2122 non-null   float64
 9   price_per_sqft            2122 non-null   float64
 10  area_type_Super_built_up  2122 non-null   int64  
 11  area_type_Built_up        2122 non-null   int64  
 12  area_type_Carpet          2122 non-null   int64  
dtypes: float64(8), int64(3), object(2)
memory usage: 232.1+ KB


In [59]:
test.head()

,ID,area_type,availability,location,size,society,total_sqft,bath,balcony
0,0,Super built-up Area,Ready To Move,Chamrajpet,2 BHK,NaN,650,1.000000,1.000000
1,1,Super built-up Area,Ready To Move,7th Phase JP Nagar,3 BHK,SrncyRe,1370,2.000000,1.000000
2,2,Super built-up Area,Ready To Move,Whitefield,3 BHK,AjhalNa,1725,3.000000,2.000000
3,3,Built-up Area,Ready To Move,Jalahalli,2 BHK,NaN,1000,2.000000,0.000000
4,4,Plot Area,Ready To Move,TC Palaya,1 Bedroom,NaN,1350,1.000000,0.000000


In [60]:
train.drop(['price_per_sqft'], axis=1, inplace=True)

x = train.drop('price', axis=1)
y = train['price']

x = x.select_dtypes(include= 'number')
x = sm.add_constant(x)

X_train, X_test, y_train, y_test = train_test_split(x, y, random_state=1, test_size = 0.2)

# xTest = test.drop('price', axis=1)
# yTest = test['price']

In [61]:
myModel = sm.OLS(y_train, X_train).fit()
myModel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.389
Model:                            OLS   Adj. R-squared:                  0.386
Method:                 Least Squares   F-statistic:                     119.4
Date:                Thu, 15 Jan 2026   Prob (F-statistic):          1.42e-173
Time:                        13:29:25   Log-Likelihood:                -1966.4
No. Observations:                1697   AIC:                             3953.
Df Residuals:                    1687   BIC:                             4007.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                        0.4694      0.030     15.521      0.000       0.410       0.529
ID                           0.0074      0.019      0.397      0.691      -0.029       0.044
availability                -0.0295      0.019     -1.559      0.119      -0.067       0.008
size                        -0.1182      0.043     -2.754      0.006      -0.202      -0.034
total_sqft                   0.0135      0.018      0.733      0.464      -0.023       0.050
bath                         0.3602      0.044      8.237      0.000       0.274       0.446
balcony                      0.1616      0.020      8.019      0.000       0.122       0.201
area_type_Super_built_up    -0.8957      0.048    -18.854      0.000      -0.989      -0.803
area_type_Built_up          -0.8102      0.057    -14.116      0.000      -0.923      -0.698
area_type_Carpet            -0.9950      0.261     -3.808      0.000      -1.507      -0.482
==============================================================================
Omnibus:                     1872.590   Durbin-Watson:                   1.927
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           255057.589
Skew:                           5.268   Prob(JB):                         0.00
Kurtosis:                      62.128   Cond. No.                         20.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [67]:
# initiate linear regression model to use in feature selection
linreg = LinearRegression()

# build forward feature selection
# pass the regression model to 'estimator'
# pass number of required feartures to 'k_features'. Here '12' is the stopping rule
# 'forward=True' performs forward selection method
# 'verbose=1' returns the number of features at the corresponding step
# 'verbose=2' returns the R-squared scores and the number of features at the corresponding step
# 'scoring=r2' considers R-squared score to select the feature
# linreg_forward = sfs(estimator=linreg, k_features = 12, forward=True,
#                      verbose=2, scoring='r2')

# fit the forward selection on training data using fit()
# sfs_forward = linreg_forward.fit(X_train, y_train)


linreg.fit(X_train, y_train)

print(f"Model Score: {model.score(X_test, y_test)}")

NameError: name 'model' is not defined